In [0]:
import os
from openai import OpenAI
from databricks.ai_search.client import AISearchClient
import uuid
import time
from datetime import datetime, timezone
from pyspark.sql.types import *

QUESTION = "What is the history of the University of Notre Dame?"
IMPOSSIBLE_QUESTION = "How much DPS does Mavuika's strongest team do in Genshin Impact?"

DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")

DATABRICKS_TOKEN = dbutils.secrets.get(
    scope="artem-gulidov-scope",
    key="artem-gulidov-pat"
)

llm_client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{DATABRICKS_HOST}/ai-gateway/mlflow/v1"
)

MODEL_NAME = "system.ai.gpt-oss-120b"

ai_search_client = AISearchClient()

INDEX_NAME = "dbr_dev_ua5816bd.gulidov_artem_silver.rag_squad_index"

index = ai_search_client.get_index(index_name=INDEX_NAME)

LOG_TABLE = "dbr_dev_ua5816bd.gulidov_artem_silver.rag_query_logs"

RAG_PROMPT = """
    You are a question-answering assistant.

    Your task is to answer the question using ONLY the
    information explicitly supported by the provided context.

    Rules:
    1. Do not use external knowledge or assumptions aside from what is provided to you in the context.
    2. Do not guess or infer facts that are not explicitly supported by the context.
    3. If the context does not contain sufficient information to answer the question, return exactly:
    I don't have enough information to answer that question.

    Output only the answer. Do not include explanations of your reasoning or any additional commentary.
"""

NO_RAG_PROMPT = """
    You are a question-answering assistant.

    Answer the user's question as accurately as possible
    using your existing knowledge.

    If you don't know the answer, say so.
    Do not invent facts.
"""

In [0]:
def retrieve_context(question, num_results=5):
    results = index.similarity_search(
        query_text=question,
        columns=[
            "chunk_id",
            "document_id",
            "title",
            "chunk_text"
        ],
        num_results=num_results
    )

    column_names = [
        column["name"]
        for column in results["manifest"]["columns"]
    ]

    retrieved_chunks = [
        dict(zip(column_names, row))
        for row in results["result"]["data_array"]
    ]

    return retrieved_chunks

In [0]:
def answer_question(question):
    total_start = time.perf_counter()

    chunks = []
    answer = None
    prompt = None
    response = None
    retrieval_ms = 0
    generation_ms = 0
    error = None

    try:
        start = time.perf_counter()

        chunks = retrieve_context(question, num_results=5)

        retrieval_ms = int((time.perf_counter() - start) * 1000)

        context = "\n\n".join(
            [
                f"Source {i}: {chunk['title']}\n{chunk['chunk_text']}"
                for i, chunk in enumerate(chunks, 1)
            ]
        )

        prompt = f"""
            Context:
            {context}

            Question:
            {question}
        """

        start = time.perf_counter()

        response = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": RAG_PROMPT},
                {"role": "user", "content": prompt}
            ],
            max_tokens=1500,
            temperature=0
        )

        generation_ms = int((time.perf_counter() - start) * 1000)

        answer = response.choices[0].message.content

        if not answer or not answer.strip():
            answer = "Empty response"

    except Exception as e:
        error = str(e)
        answer = "An error occurred while processing the question."

    finally:
        total_ms = int((time.perf_counter() - total_start) * 1000)

        log_request(
            question=question,
            prompt=prompt,
            response=answer,
            chunks=chunks,
            model_name=MODEL_NAME,
            latency_ms=total_ms,
            retrieval_latency_ms=retrieval_ms,
            generation_latency_ms=generation_ms,
            usage=response.usage if response else None,
            finish_reason=(
                response.choices[0].finish_reason
                if response else None
            ),
            used_rag=True,
            error=error
        )

    if error:
        raise RuntimeError(error)

    return {
        "question": question,
        "answer": answer,
        "sources": [
            {
                "title": c["title"],
                "chunk_id": c["chunk_id"]
            }
            for c in chunks
        ],
        "latency_ms": total_ms
    }

In [0]:
def log_request(
    question,
    prompt,
    response,
    chunks,
    model_name,
    latency_ms,
    retrieval_latency_ms,
    generation_latency_ms,
    usage,
    finish_reason,
    used_rag,
    error=None
):

    input_tokens = (
        usage.prompt_tokens if usage else None
    )

    output_tokens = (
        usage.completion_tokens if usage else None
    )

    log_data = [(
        str(uuid.uuid4()),
        question,
        prompt,
        response,
        [c["chunk_id"] for c in chunks],
        [c["title"] for c in chunks],
        len(chunks),
        model_name,
        latency_ms,
        retrieval_latency_ms,
        generation_latency_ms,
        input_tokens,
        output_tokens,
        finish_reason,
        used_rag,
        datetime.now(timezone.utc).replace(tzinfo=None),
        error
    )]

    log_schema = StructType([
        StructField("request_id", StringType()),
        StructField("question", StringType()),
        StructField("prompt", StringType()),
        StructField("response", StringType()),
        StructField("retrieved_chunk_ids", ArrayType(StringType())),
        StructField("retrieved_titles", ArrayType(StringType())),
        StructField("num_chunks", IntegerType()),
        StructField("model_name", StringType()),
        StructField("latency_ms", LongType()),
        StructField("retrieval_latency_ms", LongType()),
        StructField("generation_latency_ms", LongType()),
        StructField("input_tokens", LongType()),
        StructField("output_tokens", LongType()),
        StructField("finish_reason", StringType()),
        StructField("used_rag", BooleanType()),
        StructField("created_at", TimestampType()),
        StructField("error", StringType())
    ])

    log_df = spark.createDataFrame(log_data, schema=log_schema)

    log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)

In [0]:
import time

def answer_no_rag(question):

    start_time = time.perf_counter()

    answer = None
    prompt = question
    response = None
    error = None
    generation_ms = 0

    try:
        start = time.perf_counter()

        response = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": NO_RAG_PROMPT
                },
                {
                    "role": "user",
                    "content": question
                }
            ],
            max_tokens=1000,
            temperature=0.1
        )

        generation_ms = int(
            (time.perf_counter() - start) * 1000
        )

        answer = response.choices[0].message.content

        if not answer or not answer.strip():
            answer = "Empty response"

    except Exception as e:
        error = str(e)
        answer = "An error occurred while processing the question."

    finally:
        total_ms = int(
            (time.perf_counter() - start_time) * 1000
        )

        log_request(
            question=question,
            prompt=prompt,
            response=answer,
            chunks=[],
            model_name=MODEL_NAME,
            latency_ms=total_ms,
            retrieval_latency_ms=0,
            generation_latency_ms=generation_ms,
            usage=response.usage if response else None,
            finish_reason=(
                response.choices[0].finish_reason
                if response else None
            ),
            used_rag=False,
            error=error
        )

    if error:
        raise RuntimeError(error)

    return answer.strip()

In [0]:
#possible question, RAG
result = answer_question(QUESTION)

print("Question:", result["question"])
print("\nAnswer:", result["answer"])
print("\nSources:")

for source in result["sources"]:
    print(source)

In [0]:
#possible question, no RAG
result = answer_no_rag(QUESTION)

print("Question:", QUESTION)
print("\nAnswer:", result)

In [0]:
#impossible question, RAG
result = answer_question(IMPOSSIBLE_QUESTION)

print("Question:", result["question"])
print("\nAnswer:", result["answer"])
print("\nSources:")

for source in result["sources"]:
    print(source)

In [0]:
#impossible question, no RAG
result = answer_no_rag(IMPOSSIBLE_QUESTION)

print("Question:", IMPOSSIBLE_QUESTION)
print("\nAnswer:", result)

In [0]:
%skip
%sql
GRANT USE CATALOG
ON CATALOG dbr_dev_ua5816bd
TO `rag_readers`;

GRANT USE SCHEMA
ON SCHEMA dbr_dev_ua5816bd.gulidov_artem_silver
TO `rag_readers`;

GRANT SELECT
ON TABLE dbr_dev_ua5816bd.gulidov_artem_silver.document_chunks
TO `rag_readers`;

GRANT SELECT
ON TABLE dbr_dev_ua5816bd.gulidov_artem_silver.document_chunks
TO `rag_readers`;